# 🧭 Bússola Pública — Etapa 1: Exploração da API da Câmara dos Deputados

**Objetivo:** entender o terreno antes de escrever código de produção.  
**Não é para produção.** É para responder perguntas:
- O que cada endpoint retorna?
- Como funciona a paginação?
- Quais campos vêm nulos?
- Quais campos são chaves de relacionamento entre tabelas?

**Documentação oficial:** https://dadosabertos.camara.leg.br/swagger/api.html  
**Base URL:** `https://dadosabertos.camara.leg.br/api/v2`


# 0. Setup

In [1]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)   # mostra todas as colunas
pd.set_option('display.max_colwidth', 80)    # não trunca texto nas células

BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"
HEADERS  = {"Accept": "application/json"}

# Recorte temporal: últimos 30 dias
DATA_FIM    = datetime.today().strftime("%Y-%m-%d")
DATA_INICIO = (datetime.today() - timedelta(days=30)).strftime("%Y-%m-%d")

print(f"Período de análise: {DATA_INICIO} → {DATA_FIM}")

Período de análise: 2026-04-17 → 2026-05-17


## 1. Utilitário de chamada

Antes de explorar cada endpoint, criamos uma função auxiliar simples.  
Ela **não** é o código de produção — é só para não repetir boilerplate no notebook.

In [2]:
def get_endpoint(path: str, params: dict = {}) -> dict:
    """
    Faz uma chamada GET ao endpoint e retorna o JSON completo.
    Imprime status e quantidade de itens retornados para diagnóstico rápido.
    """
    url = f"{BASE_URL}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
    
    print(f"[{resp.status_code}] {resp.url}")
    
    if resp.status_code != 200:
        print(f"  ⚠️  Erro: {resp.text[:200]}")
        return {}
    
    data = resp.json()
    n_items = len(data.get("dados", []))
    print(f"  → {n_items} itens retornados nessa página")
    return data


def normalizar(data: dict) -> pd.DataFrame:
    """
    Converte o campo 'dados' da resposta em um DataFrame normalizado.
    """
    return pd.json_normalize(data.get("dados", []))

---
## 2. Explorando `/partidos`

Começo pelos partidos porque:
- Volume pequeno (poucos registros)
- Não precisa de filtro de data
- É uma tabela dimensão que vai se relacionar com deputados

In [19]:
# Primeira chamada — página 1, poucos itens para ver a estrutura

raw_partidos = get_endpoint("/partidos", params={"pagina": 1,
                                                 'itens': 100, 
                                                 "ordem": "ASC", 
                                                 "ordenarPor": "sigla"
                                                 })
# Inspeciona a estrutura bruta da resposta
print("\nChaves do JSON raíz:", list(raw_partidos.keys()))

[200] https://dadosabertos.camara.leg.br/api/v2/partidos?pagina=1&itens=100&ordem=ASC&ordenarPor=sigla
  → 21 itens retornados nessa página

Chaves do JSON raíz: ['dados', 'links']


In [5]:
#Inspeciona os 'links' - a API usa HATEOAS para paginação
print("Links de paginação:")
for link in raw_partidos.get("links", []):
    print(f" {link}")

Links de paginação:
 {'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?pagina=1&itens=10&ordem=ASC&ordenarPor=sigla'}
 {'rel': 'next', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?ordem=ASC&ordenarPor=sigla&pagina=2&itens=10'}
 {'rel': 'first', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?ordem=ASC&ordenarPor=sigla&pagina=1&itens=10'}
 {'rel': 'last', 'href': 'https://dadosabertos.camara.leg.br/api/v2/partidos?ordem=ASC&ordenarPor=sigla&pagina=3&itens=10'}


In [7]:
df_partidos = normalizar(raw_partidos)
print(f'Shape: {df_partidos.shape}')
df_partidos.head(10)

Shape: (10, 4)


,id,sigla,nome,uri
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/partidos/36898
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/partidos/37905
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/partidos/36899
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/partidos/38011
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/partidos/37901
5,36779,PCdoB,Partido Comunista do Brasil,https://dadosabertos.camara.leg.br/api/v2/partidos/36779
6,36786,PDT,Partido Democrático Trabalhista,https://dadosabertos.camara.leg.br/api/v2/partidos/36786
7,37906,PL,Partido Liberal,https://dadosabertos.camara.leg.br/api/v2/partidos/37906
8,36896,PODE,Podemos,https://dadosabertos.camara.leg.br/api/v2/partidos/36896
9,37903,PP,Progressistas,https://dadosabertos.camara.leg.br/api/v2/partidos/37903


In [8]:
#Análise de nulos - Campos que vem vazios com frequência
print('Campos e % de nulos:')
print(df_partidos.isnull().mean().sort_values(ascending=False).to_string())

Campos e % de nulos:
id       0.0
sigla    0.0
nome     0.0
uri      0.0


In [9]:
# Olha um registro completo para entender todos os campos disponíveis
print("Exemplo de um partido (raw JSON):")
print(json.dumps(raw_partidos["dados"][0], indent=2, ensure_ascii=False))

Exemplo de um partido (raw JSON):
{
  "id": 36898,
  "sigla": "AVANTE",
  "nome": "Avante",
  "uri": "https://dadosabertos.camara.leg.br/api/v2/partidos/36898"
}


### 📝 Observações sobre `/partidos`

> - Campos disponíveis: `id`, `sigla`, `nome`, `uri`
> - Campos nulos: `0`
> - Chave de relacionamento com deputados: `sigla` ou `id`
> - Volume total (precisa de paginação?): `21 itens totais, não precisa de páginação se usarmos o rate maximo 100`
> - Decisão: quais campos vou persistir no banco? `id`, `sigla`, `nome`

---
## 3. Explorando `/deputados`

Os 513 deputados em exercício. 

In [21]:
# Primeira página de deputados
raw_deputados = get_endpoint("/deputados", params={"pagina": 1, "itens": 100, "ordem": "ASC", "ordenarPor": "nome"})

df_deputados = normalizar(raw_deputados)
print(f"Shape (1 página): {df_deputados.shape}")
df_deputados.head()

[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100&ordem=ASC&ordenarPor=nome
  → 100 itens retornados nessa página
Shape (1 página): (100, 9)


,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,204379,https://dadosabertos.camara.leg.br/api/v2/deputados/204379,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36899,AP,57,https://www.camara.leg.br/internet/deputado/bandep/204379.jpg,dep.acaciofavacho@camara.leg.br
1,220714,https://dadosabertos.camara.leg.br/api/v2/deputados/220714,Adail Filho,MDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36899,AM,57,https://www.camara.leg.br/internet/deputado/bandep/220714.jpg,dep.adailfilho@camara.leg.br
2,221328,https://dadosabertos.camara.leg.br/api/v2/deputados/221328,Adilson Barroso,PL,https://dadosabertos.camara.leg.br/api/v2/partidos/37906,SP,57,https://www.camara.leg.br/internet/deputado/bandep/221328.jpg,dep.adilsonbarroso@camara.leg.br
3,204560,https://dadosabertos.camara.leg.br/api/v2/deputados/204560,Adolfo Viana,PSDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36835,BA,57,https://www.camara.leg.br/internet/deputado/bandep/204560.jpg,dep.adolfoviana@camara.leg.br
4,204528,https://dadosabertos.camara.leg.br/api/v2/deputados/204528,Adriana Ventura,NOVO,https://dadosabertos.camara.leg.br/api/v2/partidos/37901,SP,57,https://www.camara.leg.br/internet/deputado/bandep/204528.jpg,dep.adrianaventura@camara.leg.br


In [22]:
# Inspeciona os tipos de dado de cada coluna
print("Tipos de dado:")
print(df_deputados.dtypes.to_string())

Tipos de dado:
id               int64
uri                str
nome               str
siglaPartido       str
uriPartido         str
siglaUf            str
idLegislatura    int64
urlFoto            str
email              str


In [27]:
# PAGINAÇÃO — entendendo quantas páginas existem
# Pega uma página grande para estimar o total
teste_pag = get_endpoint("/deputados", params={"pagina": 1, "itens": 100})
n_pagina_1 = len(teste_pag.get("dados", []))
print(f"Itens na página 1 (max=100): {n_pagina_1}")

teste_pag2 = get_endpoint("/deputados", params={"pagina": 2, "itens": 100})
n_pagina_2 = len(teste_pag2.get("dados", []))
print(f"Itens na página 2 (max=100): {n_pagina_2}")

teste_pag2 = get_endpoint("/deputados", params={"pagina": 6, "itens": 100})
n_pagina_2 = len(teste_pag2.get("dados", []))
print(f"Itens na página 2 (max=100): {n_pagina_2}")

print(f"\nTotal estimado: ~{n_pagina_1 + n_pagina_2} deputados")
print("Conclusão: precisa de paginação?",(n_pagina_2 > 0))

[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100
  → 100 itens retornados nessa página
Itens na página 1 (max=100): 100
[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=2&itens=100
  → 100 itens retornados nessa página
Itens na página 2 (max=100): 100
[200] https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=6&itens=100
  → 13 itens retornados nessa página
Itens na página 2 (max=100): 13

Total estimado: ~113 deputados
Conclusão: precisa de paginação? True


In [ ]:
# Análise de nulos
print("Campos e % de nulos:")
print(df_deputados.isnull().mean().sort_values(ascending=False).to_string())

# Verifica a distribuição por partido (útil para validar depois)
if "siglaPartido" in df_deputados.columns:
    print("Deputados por partido (amostra 1 página):")
    print(df_deputados["siglaPartido"].value_counts().to_string())

# Verifica a distribuição por UF
if "siglaUf" in df_deputados.columns:
    print("Deputados por UF (amostra 1 página):")
    print(df_deputados["siglaUf"].value_counts().to_string())

Campos e % de nulos:
id               0.0
uri              0.0
nome             0.0
siglaPartido     0.0
uriPartido       0.0
siglaUf          0.0
idLegislatura    0.0
urlFoto          0.0
email            0.0
Deputados por partido (amostra 1 página):
siglaPartido
PL               18
PP               15
PT               14
MDB              12
REPUBLICANOS      8
PSD               6
UNIÃO             5
PSDB              4
PV                4
PDT               3
CIDADANIA         2
PODE              2
PSOL              2
NOVO              1
PCdoB             1
REDE              1
PSB               1
SOLIDARIEDADE     1
Deputados por UF (amostra 1 página):
siglaUf
SP    16
BA     9
RS     8
RJ     8
MG     6
PE     5
AM     4
PA     4
CE     4
PR     4
MA     4
SC     4
AP     3
TO     3
GO     2
PB     2
DF     2
AL     2
PI     2
RN     2
MS     2
RR     1
ES     1
AC     1
MT     1


In [25]:
# Detalhes de UM deputado — o endpoint /deputados/{id} tem muito mais campos
id_exemplo = df_deputados["id"].iloc[0]
raw_detalhe = get_endpoint(f"/deputados/{id_exemplo}")

print("\nCampos disponíveis no detalhe do deputado:")
print(json.dumps(raw_detalhe.get("dados", {}), indent=2, ensure_ascii=False))

[200] https://dadosabertos.camara.leg.br/api/v2/deputados/204379
  → 13 itens retornados nessa página

Campos disponíveis no detalhe do deputado:
{
  "id": 204379,
  "uri": "https://dadosabertos.camara.leg.br/api/v2/deputados/204379",
  "nomeCivil": "ACÁCIO DA SILVA FAVACHO NETO",
  "ultimoStatus": {
    "id": 204379,
    "uri": "https://dadosabertos.camara.leg.br/api/v2/deputados/204379",
    "nome": "Acácio Favacho",
    "siglaPartido": "MDB",
    "uriPartido": null,
    "siglaUf": "AP",
    "idLegislatura": 57,
    "urlFoto": "https://www.camara.leg.br/internet/deputado/bandep/204379.jpg",
    "email": null,
    "data": "2023-02-01",
    "nomeEleitoral": "Acácio Favacho",
    "gabinete": {
      "nome": "414",
      "predio": "4",
      "sala": "414",
      "andar": "4",
      "telefone": "3215-5414",
      "email": "dep.acaciofavacho@camara.leg.br"
    },
    "situacao": "Exercício",
    "condicaoEleitoral": "Titular",
    "descricaoStatus": null
  },
  "cpf": "74287028287",
  "sex

In [26]:
#Inspeciona os 'links' - a API usa HATEOAS para paginação
print("Links de paginação:")
for link in raw_deputados.get("links", []):
    print(f" {link}")

Links de paginação:
 {'rel': 'self', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100'}
 {'rel': 'next', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=2&itens=100'}
 {'rel': 'first', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=100'}
 {'rel': 'last', 'href': 'https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=6&itens=100'}


### 📝 Observações sobre `/deputados`

> - Campos no endpoint de lista: `id`, `uri`, `nome`, `siglaPartido`, `uriPartido`, `siglaUf`, `idLegislatura`, `urlFoto` `email`              `
> - Campos adicionais no endpoint de detalhe (`/deputados/{id}`): "id": 204379,
  `uri`,
  `nomeCivil`,
  `ultimoStatus`,
    `id`,
    `uri`,
    `nome`,
    `siglaPartido`,
    `uriPartido`,
    `siglaUf`,
    `idLegislatura`,
    `urlFoto`,
    `email`,
    `data`,
    `nomeEleitoral`,
    `gabinete`,
      `nome`,
      `predio`,
      `sala`,
      `andar`,
      `telefone`,
      `email`,
    `situacao`,
    `condicaoEleitoral`,
    `descricaoStatus`,
  `cpf`,
  `sexo`,
  `urlWebsite`,
  `redeSocial`,
  `dataNascimento`,
  `dataFalecimento`,
  `ufNascimento`,
  `municipioNascimento`,
  `escolaridade`
> - Total de deputados: `513`
> - Quantas páginas com `itens=100`: `6 páginas, sendo 5 com 100 itens e a sexta com 13`
> - Campos nulos recorrentes: `0`
> - Chave de relacionamento para outras tabelas: `id` (inteiro)
> - Decisão: vou usar lista ou detalhe? Lista é suficiente ou preciso de ambos? `Precisaremos de ambos. Algumas informações como a situação atual do deputa é importante e está nos detalhes`